# RS2-Net + T1-guided superior-skull refinement experiment

This notebook takes the same frozen 10 pre-Gd T1 images used by the primary
benchmark and produces four directly comparable **automatic pre-labels**:

1. `rs2net_raw` — untouched RS2-Net output;
2. `rs2_m_seam` — direct cut along a detected dark M-shaped superior gap;
3. `rs2_marker_watershed` — marker-controlled watershed near that gap;
4. `rs2_random_walker` — marker-based random walker near that gap.

The correction methods never add foreground. They search only near the superior
RS2 boundary and leave a slice unchanged when the image does not supply a
sufficiently continuous dark gap. These safeguards do **not** make the masks
ground truth: every candidate still requires visual review, especially for
cortical under-segmentation. Non-destructive 3-D regularity checks also flag
discontinuous area profiles, centroid jumps, isolated one-slice changes, and
physically irregular surfaces for closer review.

## Run instructions

1. In Colab choose **Runtime → Change runtime type → T4 GPU**.
2. Run every cell in order.
3. Upload `t1_brain_extraction_benchmark_10.zip` when prompted.
4. Use the interactive viewer and saved montages to compare all four masks.
5. Download `t1_brain_extraction_rs2_refinement_results.zip` from the final cell.

The untouched RS2 mask is preserved. Correction outputs are experimental and
separately named; nothing is silently selected as the winner.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
import torch

RS2_USE_TTA = True
RUN_RANDOM_WALKER = True
RS2_COMMIT = '144b032df4885a3da00e0d1824fdd777b3cd304f'
RS2_REPOSITORY = 'https://github.com/VitoLin21/Rodent-Skull-Stripping.git'
RS2_DRIVE_FOLDER_ID = '1cTlFFGL9iTUoZOT5Rgqi2ZAyqyPlXYd-'

BASE = Path('/content/lys_brain_refinement')
PACKAGE_AREA = BASE / 'package'
EXTERNAL = BASE / 'external'
WORK = BASE / 'work'
RESULTS = BASE / 't1_brain_extraction_rs2_refinement_results'
for directory in (PACKAGE_AREA, EXTERNAL, WORK, RESULTS):
    directory.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. Select Runtime → Change runtime type → T4 GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__)


In [ ]:
# Upload and unpack the already prepared frozen ten-image package.
from google.colab import files
import csv, json, zipfile

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Upload exactly one .zip package; received: {list(uploaded)}')
upload_path = Path('/content') / zip_names[0]
if PACKAGE_AREA.exists():
    shutil.rmtree(PACKAGE_AREA)
PACKAGE_AREA.mkdir(parents=True)
with zipfile.ZipFile(upload_path) as archive:
    for member in archive.infolist():
        target = (PACKAGE_AREA / member.filename).resolve()
        if not target.is_relative_to(PACKAGE_AREA.resolve()):
            raise ValueError(f'Unsafe archive path: {member.filename}')
    archive.extractall(PACKAGE_AREA)

manifests = list(PACKAGE_AREA.rglob('benchmark_manifest.csv'))
if len(manifests) != 1:
    raise ValueError(f'Expected one benchmark_manifest.csv, found {len(manifests)}')
PACKAGE_ROOT = manifests[0].parent
with manifests[0].open(newline='') as stream:
    PACKAGE_ROWS = list(csv.DictReader(stream))
if len(PACKAGE_ROWS) != 10:
    raise ValueError(f'This experiment expects exactly 10 cases, found {len(PACKAGE_ROWS)}')

if RESULTS.exists():
    shutil.rmtree(RESULTS)
(RESULTS / 'inputs').mkdir(parents=True)
CASES = []
for row in PACKAGE_ROWS:
    source = PACKAGE_ROOT / row['image']
    destination = RESULTS / 'inputs' / f"{row['case_id']}_pre_t1.nii.gz"
    shutil.copy2(source, destination)
    CASES.append({'case_id': row['case_id'], 'image': destination})
shutil.copy2(manifests[0], RESULTS / 'input_benchmark_manifest.csv')
if (PACKAGE_ROOT / 'package_metadata.json').is_file():
    shutil.copy2(PACKAGE_ROOT / 'package_metadata.json', RESULTS / 'input_package_metadata.json')
print('Cases:', ', '.join(case['case_id'] for case in CASES))


In [ ]:
# Install the known-working RS2 environment, pin its source, and fetch official weights.
packages = [
    'monai==1.4.0', 'nibabel>=5.3,<6', 'nilearn>=0.12,<1',
    'scikit-image>=0.23,<1', 'einops>=0.8,<1', 'gdown>=5.2,<6',
    'acvl_utils==0.2.1', 'batchgenerators>=0.25,<1',
    'SimpleITK>=2.4,<3', 'tifffile', 'imageio', 'pandas', 'ipywidgets'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])
runtime_preflight = (
    "import monai, torch; "
    "assert monai.__version__.startswith('1.4.'), monai.__version__; "
    "assert torch.cuda.is_available(), 'GPU unavailable to subprocess'; "
    "print('Runtime preflight: MONAI', monai.__version__, '| GPU', torch.cuda.get_device_name(0))"
)
subprocess.check_call([sys.executable, '-c', runtime_preflight])

def clone_pinned(repository, commit, destination):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.check_call(['git', 'clone', '-q', repository, str(destination)])
    subprocess.check_call(['git', '-C', str(destination), 'checkout', '-q', commit])
    actual = subprocess.check_output(
        ['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True
    ).strip()
    if actual != commit:
        raise RuntimeError(f'Pinned checkout failed: expected {commit}, got {actual}')

RS2_ROOT = EXTERNAL / 'Rodent-Skull-Stripping'
clone_pinned(RS2_REPOSITORY, RS2_COMMIT, RS2_ROOT)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(RS2_ROOT), '--no-deps'])

# PyTorch >=2.6 defaults torch.load to weights_only=True. The official legacy
# checkpoint contains NumPy metadata, so explicitly allow the trusted official file.
RS2_PREDICT_SOURCE = RS2_ROOT / 'RS2' / 'inference' / 'predict.py'
rs2_source = RS2_PREDICT_SOURCE.read_text()
old_load = "torch.load(checkpoint_name, map_location=torch.device('cpu'))"
new_load = "torch.load(checkpoint_name, map_location=torch.device('cpu'), weights_only=False)"
if new_load not in rs2_source:
    if rs2_source.count(old_load) != 1:
        raise RuntimeError('Could not apply the pinned RS2 PyTorch compatibility patch')
    RS2_PREDICT_SOURCE.write_text(rs2_source.replace(old_load, new_load))

model_preflight = '''
from RS2.network.RSSNet import RSSNet
model = RSSNet(img_size=(128, 128, 160), in_channels=1, out_channels=1, feature_size=48)
del model
print('RS2 model preflight passed')
'''
preflight_env = os.environ.copy()
preflight_env['PYTHONPATH'] = os.pathsep.join([str(RS2_ROOT), preflight_env.get('PYTHONPATH', '')])
subprocess.check_call([sys.executable, '-c', model_preflight], env=preflight_env)

import gdown
RS2_WEIGHTS_AREA = EXTERNAL / 'rs2_weights'
RS2_WEIGHTS_AREA.mkdir(parents=True, exist_ok=True)
if not list(RS2_WEIGHTS_AREA.rglob('*pretrained_model.pt')):
    print('Downloading official RS2-Net weights...')
    gdown.download_folder(
        id=RS2_DRIVE_FOLDER_ID, output=str(RS2_WEIGHTS_AREA),
        quiet=False, use_cookies=False
    )
rs2_candidates = list(RS2_WEIGHTS_AREA.rglob('*pretrained_model.pt'))
if len(rs2_candidates) != 1:
    raise FileNotFoundError(f'Expected one RS2 checkpoint, found {rs2_candidates}')
RS2_WEIGHT = rs2_candidates[0]
checkpoint_preflight = (
    "import torch; "
    f"checkpoint=torch.load({str(RS2_WEIGHT)!r}, map_location='cpu', weights_only=False); "
    "assert isinstance(checkpoint, dict) and 'state_dict' in checkpoint; "
    "print('RS2 checkpoint preflight: state_dict found')"
)
subprocess.check_call([sys.executable, '-c', checkpoint_preflight])


In [ ]:
# Output contract, provenance, orientation, and NIfTI helpers.
import hashlib, importlib.metadata, platform, time
from datetime import datetime, timezone
import nibabel as nib
from nibabel.processing import resample_from_to
import numpy as np

RUN_FIELDS = ['case_id', 'model_id', 'status', 'image', 'mask', 'metadata', 'log', 'message']
RUN_ROWS = {}

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def relative(path):
    return str(Path(path).resolve().relative_to(RESULTS.resolve())) if path else ''

def write_run_manifest():
    with (RESULTS / 'run_manifest.csv').open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=RUN_FIELDS)
        writer.writeheader()
        writer.writerows(RUN_ROWS[key] for key in sorted(RUN_ROWS))

def record(case_id, model_id, status, image, mask=None, metadata=None, log=None, message=''):
    RUN_ROWS[(case_id, model_id)] = {
        'case_id': case_id, 'model_id': model_id, 'status': status,
        'image': relative(image), 'mask': relative(mask),
        'metadata': relative(metadata), 'log': relative(log), 'message': message,
    }
    write_run_manifest()

def run_logged(command, log_path, cwd=None):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('$', ' '.join(str(part) for part in command))
    with log_path.open('w') as log:
        process = subprocess.Popen(
            [str(part) for part in command], cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, bufsize=1
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        return process.wait()

def standardize_mask(source, reference, destination):
    reference_image = nib.load(str(reference))
    mask_image = nib.load(str(source))
    resampled = False
    if mask_image.shape != reference_image.shape or not np.allclose(
        mask_image.affine, reference_image.affine, rtol=1e-5, atol=1e-5
    ):
        mask_image = resample_from_to(mask_image, reference_image, order=0)
        resampled = True
    mask = (np.asanyarray(mask_image.dataobj) > 0).astype(np.uint8)
    save_mask_like(mask, reference_image, destination)
    return {'resampled_to_input_grid': resampled, 'foreground_voxels': int(mask.sum())}

def save_mask_like(mask, reference_image, destination):
    mask = np.asarray(mask, dtype=np.uint8)
    if mask.shape != reference_image.shape or not mask.any() or mask.all():
        raise ValueError(f'Invalid mask for {destination}: shape={mask.shape}, foreground={int(mask.sum())}')
    header = reference_image.header.copy()
    header.set_data_dtype(np.uint8)
    destination.parent.mkdir(parents=True, exist_ok=True)
    output = nib.Nifti1Image(mask, reference_image.affine, header)
    output.set_qform(reference_image.affine, code=int(reference_image.header['qform_code']))
    output.set_sform(reference_image.affine, code=int(reference_image.header['sform_code']))
    nib.save(output, destination)

def native_to_rsa(array, affine):
    # Return array in R/S/A order and a reversible orientation record.
    codes = nib.aff2axcodes(affine)
    requested = [('R', 'L'), ('S', 'I'), ('A', 'P')]
    order = []
    flips = []
    for positive, negative in requested:
        matches = [index for index, code in enumerate(codes) if code in (positive, negative)]
        if len(matches) != 1:
            raise ValueError(f'Cannot identify {positive}/{negative} axis from {codes}')
        axis = matches[0]
        order.append(axis)
        flips.append(codes[axis] == negative)
    oriented = np.transpose(np.asarray(array), order)
    for axis, should_flip in enumerate(flips):
        if should_flip:
            oriented = np.flip(oriented, axis=axis)
    return oriented, {'native_axis_codes': codes, 'order': order, 'flips': flips}

def rsa_to_native(array_rsa, orientation):
    native_ordered = np.asarray(array_rsa)
    for axis, should_flip in enumerate(orientation['flips']):
        if should_flip:
            native_ordered = np.flip(native_ordered, axis=axis)
    return np.transpose(native_ordered, np.argsort(orientation['order']))

RUNTIME = {
    'created_at': utc_now(), 'python': platform.python_version(),
    'pytorch': torch.__version__, 'monai': importlib.metadata.version('monai'),
    'scikit_image': importlib.metadata.version('scikit-image'),
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
}
RS2_PROVENANCE = {
    'repository': RS2_REPOSITORY, 'commit': RS2_COMMIT,
    'weights_url': f'https://drive.google.com/drive/folders/{RS2_DRIVE_FOLDER_ID}',
    'weight_sha256': sha256(RS2_WEIGHT), 'test_time_augmentation': RS2_USE_TTA,
    'checkpoint_compatibility': 'weights_only=False for trusted official legacy checkpoint',
}


In [ ]:
# Run RS2-Net once. This raw prediction remains immutable.
rs2_input = WORK / 'rs2net_inputs'
rs2_raw = WORK / 'rs2net_outputs'
for directory in (rs2_input, rs2_raw):
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True)
for case in CASES:
    shutil.copy2(case['image'], rs2_input / f"{case['case_id']}_0000.nii.gz")

batch_log = RESULTS / 'logs' / 'rs2net_raw' / 'rs2net_batch.log'
command = [
    'RS2_predict', '-i', str(rs2_input), '-o', str(rs2_raw),
    '-m', str(RS2_WEIGHT), '-device', 'cuda', '-npp', '1', '-nps', '1',
]
if not RS2_USE_TTA:
    command.append('--disable_tta')
started = time.monotonic()
returncode = run_logged(command, batch_log, cwd=RS2_ROOT)
elapsed = round(time.monotonic() - started, 3)

for case in CASES:
    case_id, image = case['case_id'], case['image']
    mask = RESULTS / 'predictions' / 'rs2net_raw' / f'{case_id}_brain_mask.nii.gz'
    metadata = RESULTS / 'metadata' / 'rs2net_raw' / f'{case_id}.json'
    metadata.parent.mkdir(parents=True, exist_ok=True)
    candidates = [rs2_raw / f'{case_id}.nii.gz', rs2_raw / f'{case_id}_0000.nii.gz']
    source = next((path for path in candidates if path.is_file()), None)
    try:
        if source is None:
            raise FileNotFoundError(f'RS2-Net did not create an output; batch status {returncode}')
        standard = standardize_mask(source, image, mask)
        payload = {
            'case_id': case_id, 'model_id': 'rs2net_raw', 'status': 'ok',
            'role': 'immutable automatic pre-label', 'input': relative(image),
            'output': relative(mask), 'preprocessing': 'official RS2 DefaultPreprocessor',
            'batch_command': command, 'batch_runtime_seconds': elapsed,
            'input_sha256': sha256(image), 'mask_sha256': sha256(mask),
            **standard, **RS2_PROVENANCE, **RUNTIME,
        }
        metadata.write_text(json.dumps(payload, indent=2) + '\n')
        record(case_id, 'rs2net_raw', 'ok', image, mask, metadata, batch_log)
    except Exception as exc:
        message = f'{type(exc).__name__}: {exc}'
        record(case_id, 'rs2net_raw', 'failed', image, log=batch_log, message=message)
        print(f'FAILED {case_id}: {message}')

failures = [row for row in RUN_ROWS.values() if row['model_id'] == 'rs2net_raw' and row['status'] != 'ok']
if failures:
    raise RuntimeError(f'Raw RS2 prediction failed for {len(failures)} cases; stop before refinement.')
print('Raw RS2 masks ready:', len(CASES))


In [ ]:
"""Experimental T1-guided refinements for an RS2-Net brain-mask pre-label.

These functions are deliberately conservative research utilities.  They never turn an
automatic prediction into an approved mask and they leave slices unchanged when a
plausible superior brain--skull intensity valley cannot be detected.

Arrays passed to this module use anatomical R/S/A order: left-to-right, inferior-to-
superior, and posterior-to-anterior.  The Colab notebook performs and records the
orientation conversion before calling these functions.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Any, Callable

import numpy as np
from scipy import ndimage as ndi


@dataclass(frozen=True)
class GapRefinementConfig:
    """Parameters expressed primarily in physical millimetres."""

    max_search_depth_mm: float = 2.8
    min_cap_thickness_mm: float = 0.30
    valley_window_mm: float = 0.32
    line_smoothing_mm: float = 0.10
    min_valley_contrast: float = 0.10
    min_confident_width_fraction: float = 0.22
    central_fraction: float = 0.75
    max_column_jump_mm: float = 0.55
    lateral_extension_mm: float = 0.75
    extend_seam_to_mask_edges: bool = True
    seed_margin_mm: float = 0.24
    min_slice_removed_fraction: float = 0.01
    max_slice_removed_fraction: float = 0.35
    min_corrected_slices: int = 3


@dataclass(frozen=True)
class MaskRegularityConfig:
    """Conservative warning thresholds for a 3-D mouse-brain mask.

    These checks describe suspicious geometry; they never edit, accept, or reject a
    mask. End slices below ``profile_area_fraction`` of the maximum cross-sectional
    area are excluded from local-change checks because a normal brain tapers there.
    """

    profile_area_fraction: float = 0.20
    max_adjacent_area_change_fraction: float = 0.55
    max_one_slice_area_deviation_fraction: float = 0.30
    max_centroid_step_mm: float = 0.75
    min_compactness: float = 0.05


@dataclass(frozen=True)
class MSeamCleanupConfig:
    """Conservative physical rules for stabilising an M-seam draft mask.

    The cleanup is deliberately restricted to two failure modes observed during the
    frozen ten-case review: small in-plane islands in established brain-containing
    slices and short shape outlier runs bracketed by similar masks.  Interpolated
    repairs are always intersected with the immutable raw RS2 prediction, so the
    cleanup can never invent foreground outside the model prediction.
    """

    profile_area_fraction: float = 0.20
    max_secondary_component_fraction: float = 0.12
    max_secondary_component_area_mm2: float = 4.0
    consensus_half_window_mm: float = 0.70
    min_shape_disagreement_fraction: float = 0.035
    max_repair_run_mm: float = 0.60
    min_flank_dice: float = 0.92
    max_flank_area_change_fraction: float = 0.18
    max_slice_change_fraction: float = 0.12


@dataclass(frozen=True)
class MSeamCleanupReport:
    """Auditable changes made to one automatic M-seam draft mask."""

    input_foreground_voxels: int
    output_foreground_voxels: int
    disconnected_voxels_removed: int
    in_plane_island_voxels_removed: int
    in_plane_cleaned_slices: tuple[int, ...]
    candidate_outlier_slices: tuple[int, ...]
    outlier_score_by_slice: tuple[tuple[int, float], ...]
    repaired_slice_runs: tuple[tuple[int, int], ...]
    skipped_slice_runs: tuple[tuple[int, int, str], ...]
    voxels_added_from_raw_rs2: int
    voxels_removed_by_run_repair: int
    changed_voxels: int
    subset_of_raw_rs2: bool
    configuration: dict[str, Any]

    def to_dict(self) -> dict[str, Any]:
        """Return a JSON-serializable representation for provenance metadata."""

        return asdict(self)


@dataclass(frozen=True)
class MaskRegularityReport:
    """Physical and slice-profile measurements used to guide human review."""

    foreground_voxels: int
    volume_mm3: float
    connected_components: int
    occupied_slice_range: tuple[int, int]
    internal_empty_slices: tuple[int, ...]
    slice_area_mm2: tuple[float, ...]
    centroid_rs_mm: tuple[tuple[float, float] | None, ...]
    max_adjacent_area_change_fraction: float
    abrupt_area_pairs: tuple[tuple[int, int], ...]
    max_centroid_step_mm: float
    abrupt_centroid_pairs: tuple[tuple[int, int], ...]
    one_slice_outlier_slices: tuple[int, ...]
    surface_area_mm2: float
    surface_to_volume_ratio_mm_inverse: float
    compactness: float
    warnings: tuple[str, ...]

    def to_dict(self) -> dict[str, Any]:
        """Return a JSON-serializable representation for provenance metadata."""

        return asdict(self)


@dataclass
class SliceGap:
    """A detected superior separating line for one coronal slice."""

    valid: bool
    seam: np.ndarray
    x_start: int = 0
    x_stop: int = 0
    confidence: float = 0.0
    coverage: float = 0.0
    reason: str = ""


def _voxel_face_surface_area(mask: np.ndarray, spacing: np.ndarray) -> float:
    """Estimate physical surface area by counting exposed voxel faces."""

    padded = np.pad(np.asarray(mask, dtype=np.int8), 1)
    surface_area = 0.0
    for axis in range(3):
        exposed_faces = int(np.count_nonzero(np.diff(padded, axis=axis)))
        face_area = float(np.prod(np.delete(spacing, axis)))
        surface_area += exposed_faces * face_area
    return surface_area


def assess_mask_regularity(
    mask_rsa: np.ndarray,
    spacing_rsa: tuple[float, float, float],
    config: MaskRegularityConfig | None = None,
) -> MaskRegularityReport:
    """Measure 3-D regularity without changing or approving the supplied mask.

    The report looks for disconnected islands, internal empty slices, sudden changes
    in coronal area or centroid, and isolated one-slice area deviations. Surface and
    compactness measurements are expressed in physical units so anisotropic scans are
    not treated as isotropic voxel grids.
    """

    mask = np.asarray(mask_rsa, dtype=bool)
    if mask.ndim != 3:
        raise ValueError("Mask regularity assessment requires a three-dimensional mask")
    if not mask.any():
        raise ValueError("Mask regularity assessment requires a non-empty mask")
    spacing = np.asarray(spacing_rsa, dtype=np.float64)
    if spacing.shape != (3,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
        raise ValueError("Mask spacing must contain three finite positive values")
    config = config or MaskRegularityConfig()

    foreground_voxels = int(np.count_nonzero(mask))
    voxel_volume = float(np.prod(spacing))
    volume_mm3 = foreground_voxels * voxel_volume
    _, connected_components = ndi.label(mask)

    counts = np.count_nonzero(mask, axis=(0, 1)).astype(np.float64)
    areas = counts * float(spacing[0] * spacing[1])
    occupied = np.flatnonzero(counts)
    first_slice, last_slice = int(occupied[0]), int(occupied[-1])
    internal_empty = tuple(
        int(index)
        for index in range(first_slice, last_slice + 1)
        if counts[index] == 0
    )

    centroid_rs: list[tuple[float, float] | None] = []
    for slice_index in range(mask.shape[2]):
        coordinates = np.argwhere(mask[:, :, slice_index])
        if coordinates.size == 0:
            centroid_rs.append(None)
            continue
        centroid = coordinates.mean(axis=0) * spacing[:2]
        centroid_rs.append((float(centroid[0]), float(centroid[1])))

    profile = areas >= float(areas.max() * config.profile_area_fraction)
    area_changes: list[tuple[tuple[int, int], float]] = []
    centroid_steps: list[tuple[tuple[int, int], float]] = []
    for left in range(mask.shape[2] - 1):
        right = left + 1
        if not (profile[left] and profile[right]):
            continue
        denominator = max(float((areas[left] + areas[right]) / 2.0), np.finfo(float).eps)
        area_change = float(abs(areas[right] - areas[left]) / denominator)
        area_changes.append(((left, right), area_change))
        left_centroid, right_centroid = centroid_rs[left], centroid_rs[right]
        if left_centroid is not None and right_centroid is not None:
            centroid_step = float(
                np.linalg.norm(np.asarray(right_centroid) - np.asarray(left_centroid))
            )
            centroid_steps.append(((left, right), centroid_step))

    abrupt_area_pairs = tuple(
        pair
        for pair, change in area_changes
        if change > config.max_adjacent_area_change_fraction
    )
    abrupt_centroid_pairs = tuple(
        pair
        for pair, step in centroid_steps
        if step > config.max_centroid_step_mm
    )

    one_slice_outliers: list[int] = []
    for index in range(1, mask.shape[2] - 1):
        if not (profile[index - 1] and profile[index] and profile[index + 1]):
            continue
        neighbour_area = float((areas[index - 1] + areas[index + 1]) / 2.0)
        neighbour_disagreement = abs(
            float(areas[index - 1]) - float(areas[index + 1])
        ) / max(neighbour_area, np.finfo(float).eps)
        if neighbour_disagreement > config.max_one_slice_area_deviation_fraction:
            continue
        deviation = abs(float(areas[index]) - neighbour_area) / max(
            neighbour_area, np.finfo(float).eps
        )
        if deviation > config.max_one_slice_area_deviation_fraction:
            one_slice_outliers.append(index)

    surface_area = _voxel_face_surface_area(mask, spacing)
    surface_to_volume = surface_area / volume_mm3
    compactness = float(36.0 * np.pi * volume_mm3**2 / surface_area**3)

    warnings: list[str] = []
    if connected_components > 1:
        warnings.append("disconnected_components")
    if internal_empty:
        warnings.append("internal_empty_slices")
    if abrupt_area_pairs:
        warnings.append("abrupt_cross_section_change")
    if abrupt_centroid_pairs:
        warnings.append("abrupt_centroid_motion")
    if one_slice_outliers:
        warnings.append("isolated_one_slice_area_outlier")
    if compactness < config.min_compactness:
        warnings.append("low_physical_compactness")

    return MaskRegularityReport(
        foreground_voxels=foreground_voxels,
        volume_mm3=volume_mm3,
        connected_components=int(connected_components),
        occupied_slice_range=(first_slice, last_slice),
        internal_empty_slices=internal_empty,
        slice_area_mm2=tuple(float(value) for value in areas),
        centroid_rs_mm=tuple(centroid_rs),
        max_adjacent_area_change_fraction=max(
            (change for _, change in area_changes), default=0.0
        ),
        abrupt_area_pairs=abrupt_area_pairs,
        max_centroid_step_mm=max((step for _, step in centroid_steps), default=0.0),
        abrupt_centroid_pairs=abrupt_centroid_pairs,
        one_slice_outlier_slices=tuple(one_slice_outliers),
        surface_area_mm2=surface_area,
        surface_to_volume_ratio_mm_inverse=surface_to_volume,
        compactness=compactness,
        warnings=tuple(warnings),
    )


def robust_normalize(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Scale finite masked intensities to [0, 1] using robust cohort-independent limits."""

    image = np.asarray(image, dtype=np.float32)
    mask = np.asarray(mask, dtype=bool)
    values = image[mask & np.isfinite(image)]
    if values.size < 100:
        raise ValueError("Too few finite foreground voxels for robust normalization")
    low, high = np.percentile(values, (2.0, 98.0))
    if not np.isfinite(low) or not np.isfinite(high) or high <= low:
        raise ValueError(f"Invalid robust intensity limits: {low}, {high}")
    normalized = np.clip((image - low) / (high - low), 0.0, 1.0)
    normalized[~np.isfinite(normalized)] = 0.0
    return normalized.astype(np.float32, copy=False)


def _longest_true_run(values: np.ndarray) -> tuple[int, int]:
    padded = np.pad(np.asarray(values, dtype=np.int8), 1)
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1)
    if starts.size == 0:
        return 0, 0
    lengths = stops - starts
    index = int(np.argmax(lengths))
    return int(starts[index]), int(stops[index])


def detect_dark_gap(
    normalized_slice: np.ndarray,
    mask_slice: np.ndarray,
    spacing_rs: tuple[float, float],
    config: GapRefinementConfig,
) -> SliceGap:
    """Detect a smooth dark superior valley in a single R/S coronal slice."""

    image = np.asarray(normalized_slice, dtype=np.float32)
    mask = np.asarray(mask_slice, dtype=bool)
    if image.ndim != 2 or image.shape != mask.shape:
        raise ValueError("The image and mask slice must be matching two-dimensional arrays")
    occupied_x = np.flatnonzero(mask.any(axis=1))
    empty_seam = np.full(mask.shape[0], np.nan, dtype=np.float32)
    if occupied_x.size < 8:
        return SliceGap(False, empty_seam, reason="too_little_foreground")

    spacing_s = float(spacing_rs[1])
    max_depth = max(4, int(round(config.max_search_depth_mm / spacing_s)))
    min_cap = max(2, int(round(config.min_cap_thickness_mm / spacing_s)))
    window = max(2, int(round(config.valley_window_mm / spacing_s)))
    sigma = max(0.5, config.line_smoothing_mm / spacing_s)
    max_jump = max(2, int(round(config.max_column_jump_mm / spacing_s)))

    smooth = ndi.gaussian_filter1d(image, sigma=sigma, axis=1, mode="nearest")
    candidates = empty_seam.copy()
    contrasts = np.zeros(mask.shape[0], dtype=np.float32)
    tops = np.full(mask.shape[0], -1, dtype=np.int32)

    for x in occupied_x:
        ys = np.flatnonzero(mask[x])
        bottom, top = int(ys[0]), int(ys[-1])
        tops[x] = top
        lower = max(bottom + 1, top - max_depth)
        upper = top - min_cap
        if upper <= lower + 1:
            continue
        best_y = None
        best_contrast = -np.inf
        for y in range(lower, upper + 1):
            below = smooth[x, max(bottom, y - window) : y]
            above = smooth[x, y + 1 : min(top + 1, y + 1 + window)]
            if below.size < 2 or above.size < 2:
                continue
            contrast = min(float(np.mean(below)), float(np.mean(above))) - float(smooth[x, y])
            if contrast > best_contrast:
                best_y, best_contrast = y, contrast
        if best_y is not None:
            candidates[x] = best_y
            contrasts[x] = max(0.0, best_contrast)

    x0, x1 = int(occupied_x[0]), int(occupied_x[-1]) + 1
    width = x1 - x0
    central_margin = int(round(width * (1.0 - config.central_fraction) / 2.0))
    central = np.zeros(mask.shape[0], dtype=bool)
    central[x0 + central_margin : x1 - central_margin] = True
    good = np.isfinite(candidates) & (contrasts >= config.min_valley_contrast) & central
    good = ndi.binary_closing(good, structure=np.ones(5, dtype=bool))

    # Suppress isolated dark structures that jump away from the dominant superior line.
    finite = np.where(np.isfinite(candidates), candidates, 0.0)
    support = np.isfinite(candidates).astype(np.float32)
    smoothed_numerator = ndi.gaussian_filter1d(finite, sigma=2.0, mode="nearest")
    smoothed_denominator = ndi.gaussian_filter1d(support, sigma=2.0, mode="nearest")
    trend = np.divide(
        smoothed_numerator,
        smoothed_denominator,
        out=np.full_like(smoothed_numerator, np.nan),
        where=smoothed_denominator > 1e-3,
    )
    good &= np.abs(candidates - trend) <= max_jump

    run_start, run_stop = _longest_true_run(good[x0:x1])
    run_start += x0
    run_stop += x0
    run_width = run_stop - run_start
    required_width = max(6, int(round(width * config.min_confident_width_fraction)))
    if run_width < required_width:
        return SliceGap(False, empty_seam, reason="insufficient_continuous_dark_gap")

    # Interpolate between supported columns and extrapolate only a small physical margin.
    # This removes thin cap remnants at the ends of an otherwise convincing M-shaped gap
    # without extending the cut across the lateral cortex.
    support_x = np.flatnonzero(good[run_start:run_stop]) + run_start
    if config.extend_seam_to_mask_edges:
        run_start, run_stop = x0, x1
    else:
        extension = max(0, int(round(config.lateral_extension_mm / float(spacing_rs[0]))))
        run_start = max(x0, run_start - extension)
        run_stop = min(x1, run_stop + extension)
    seam = empty_seam.copy()
    seam[run_start:run_stop] = np.interp(
        np.arange(run_start, run_stop), support_x, candidates[support_x]
    )
    seam[run_start:run_stop] = ndi.median_filter(seam[run_start:run_stop], size=5, mode="nearest")
    confidence = float(np.median(contrasts[support_x]))
    coverage = float(run_width / width)
    return SliceGap(True, seam, run_start, run_stop, confidence, coverage, "ok")


def detect_gap_volume(
    normalized_rsa: np.ndarray,
    mask_rsa: np.ndarray,
    spacing_rsa: tuple[float, float, float],
    config: GapRefinementConfig,
) -> list[SliceGap]:
    """Detect candidate seams independently across posterior--anterior slices."""

    gaps = [
        detect_dark_gap(normalized_rsa[:, :, z], mask_rsa[:, :, z], spacing_rsa[:2], config)
        for z in range(mask_rsa.shape[2])
    ]
    valid = np.asarray([gap.valid for gap in gaps], dtype=bool)
    labels, count = ndi.label(valid)
    for label_id in range(1, count + 1):
        indices = np.flatnonzero(labels == label_id)
        if indices.size < config.min_corrected_slices:
            for index in indices:
                gaps[index].valid = False
                gaps[index].reason = "insufficient_continuity_across_slices"
    return gaps


def _slice_markers(
    mask_slice: np.ndarray,
    gap: SliceGap,
    spacing_s: float,
    config: GapRefinementConfig,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return brain seed, cap seed, and the local refinement domain."""

    mask = np.asarray(mask_slice, dtype=bool)
    brain = np.zeros_like(mask)
    cap = np.zeros_like(mask)
    domain = np.zeros_like(mask)
    margin = max(2, int(round(config.seed_margin_mm / spacing_s)))
    for x in range(gap.x_start, gap.x_stop):
        if not np.isfinite(gap.seam[x]):
            continue
        y = int(round(float(gap.seam[x])))
        domain[x] = mask[x]
        brain[x, : max(0, y - margin + 1)] = mask[x, : max(0, y - margin + 1)]
        cap[x, min(mask.shape[1], y + margin) :] = mask[x, min(mask.shape[1], y + margin) :]
    return brain, cap, domain


def _accept_slice(raw: np.ndarray, candidate: np.ndarray, config: GapRefinementConfig) -> bool:
    raw_count = int(np.count_nonzero(raw))
    if raw_count == 0:
        return False
    removed = int(np.count_nonzero(raw & ~candidate))
    fraction = removed / raw_count
    return config.min_slice_removed_fraction <= fraction <= config.max_slice_removed_fraction


def _largest_component(mask: np.ndarray) -> np.ndarray:
    labels, count = ndi.label(mask)
    if count <= 1:
        return np.asarray(mask, dtype=bool)
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    return labels == int(np.argmax(sizes))


def _largest_fully_connected_component(mask: np.ndarray) -> tuple[np.ndarray, int]:
    """Keep the largest component using full connectivity and report removed voxels."""

    foreground = np.asarray(mask, dtype=bool)
    structure = ndi.generate_binary_structure(foreground.ndim, foreground.ndim)
    labels, count = ndi.label(foreground, structure=structure)
    if count <= 1:
        return foreground.copy(), 0
    sizes = np.bincount(labels.ravel())
    sizes[0] = 0
    largest = labels == int(np.argmax(sizes))
    return largest, int(np.count_nonzero(foreground & ~largest))


def _prune_small_in_plane_islands(
    mask: np.ndarray,
    spacing_rs: tuple[float, float],
    config: MSeamCleanupConfig,
) -> tuple[np.ndarray, tuple[int, ...], int]:
    """Remove small secondary 2-D components only in the established brain body.

    Comparable bilateral components at the normally tapering ends are preserved.  A
    slice is changed only when the main component is within the central area profile
    and all secondary components are small both relative to it and in physical area.
    """

    output = np.asarray(mask, dtype=bool).copy()
    counts = np.count_nonzero(output, axis=(0, 1))
    if not np.any(counts):
        return output, (), 0
    body_threshold = float(counts.max() * config.profile_area_fraction)
    pixel_area_mm2 = float(spacing_rs[0] * spacing_rs[1])
    structure = ndi.generate_binary_structure(2, 2)
    cleaned_slices: list[int] = []
    removed_voxels = 0
    for slice_index in range(output.shape[2]):
        slice_mask = output[:, :, slice_index]
        if int(np.count_nonzero(slice_mask)) < body_threshold:
            continue
        labels, component_count = ndi.label(slice_mask, structure=structure)
        if component_count <= 1:
            continue
        sizes = np.bincount(labels.ravel())
        sizes[0] = 0
        largest_label = int(np.argmax(sizes))
        largest_size = int(sizes[largest_label])
        secondary_size = int(sizes.sum() - largest_size)
        if largest_size == 0 or secondary_size == 0:
            continue
        relative_size = secondary_size / largest_size
        secondary_area = secondary_size * pixel_area_mm2
        if (
            relative_size > config.max_secondary_component_fraction
            or secondary_area > config.max_secondary_component_area_mm2
        ):
            continue
        output[:, :, slice_index] = labels == largest_label
        cleaned_slices.append(slice_index)
        removed_voxels += secondary_size
    return output, tuple(cleaned_slices), removed_voxels


def _dice(left: np.ndarray, right: np.ndarray) -> float:
    left = np.asarray(left, dtype=bool)
    right = np.asarray(right, dtype=bool)
    denominator = int(np.count_nonzero(left)) + int(np.count_nonzero(right))
    if denominator == 0:
        return 1.0
    return float(2 * np.count_nonzero(left & right) / denominator)


def _signed_distance(mask: np.ndarray, spacing: tuple[float, float]) -> np.ndarray:
    foreground = np.asarray(mask, dtype=bool)
    return ndi.distance_transform_edt(foreground, sampling=spacing) - ndi.distance_transform_edt(
        ~foreground, sampling=spacing
    )


def _true_runs(values: np.ndarray) -> list[tuple[int, int]]:
    """Return inclusive start/stop indices for all true runs."""

    padded = np.pad(np.asarray(values, dtype=np.int8), 1)
    changes = np.diff(padded)
    starts = np.flatnonzero(changes == 1)
    stops = np.flatnonzero(changes == -1) - 1
    return [(int(start), int(stop)) for start, stop in zip(starts, stops, strict=True)]


def stabilize_m_seam_mask(
    m_seam_mask_rsa: np.ndarray,
    raw_rs2_mask_rsa: np.ndarray,
    spacing_rsa: tuple[float, float, float],
    config: MSeamCleanupConfig | None = None,
) -> tuple[np.ndarray, MSeamCleanupReport]:
    """Clean observed M-seam topology failures without approving the result.

    First, true disconnected 3-D islands and sufficiently small in-plane islands are
    removed.  Second, a longitudinal majority profile identifies short abnormal shape
    runs.  A run is repaired only when it is bracketed by highly similar slices; its
    masks are interpolated between the flanks using physical signed-distance fields.
    Every repair remains a subset of the immutable raw RS2 mask.
    """

    candidate = np.asarray(m_seam_mask_rsa, dtype=bool)
    raw = np.asarray(raw_rs2_mask_rsa, dtype=bool)
    if candidate.ndim != 3 or candidate.shape != raw.shape:
        raise ValueError("M-seam cleanup requires matching three-dimensional masks")
    if not candidate.any() or not raw.any():
        raise ValueError("M-seam cleanup requires non-empty masks")
    if np.any(candidate & ~raw):
        raise ValueError("The M-seam mask must be a subset of the raw RS2 prediction")
    spacing = np.asarray(spacing_rsa, dtype=np.float64)
    if spacing.shape != (3,) or not np.all(np.isfinite(spacing)) or np.any(spacing <= 0):
        raise ValueError("M-seam cleanup spacing must contain three finite positive values")
    config = config or MSeamCleanupConfig()

    input_voxels = int(np.count_nonzero(candidate))
    output, disconnected_removed = _largest_fully_connected_component(candidate)
    output, island_slices, island_removed = _prune_small_in_plane_islands(
        output, (float(spacing[0]), float(spacing[1])), config
    )
    output, newly_disconnected = _largest_fully_connected_component(output)
    disconnected_removed += newly_disconnected

    counts = np.count_nonzero(output, axis=(0, 1)).astype(np.float64)
    profile = counts >= float(counts.max() * config.profile_area_fraction)
    radius = max(2, int(np.ceil(config.consensus_half_window_mm / spacing[2])))
    scores = np.zeros(output.shape[2], dtype=np.float64)
    outliers = np.zeros(output.shape[2], dtype=bool)
    for slice_index in range(radius, output.shape[2] - radius):
        if not profile[slice_index]:
            continue
        neighbourhood = output[:, :, slice_index - radius : slice_index + radius + 1]
        consensus = np.count_nonzero(neighbourhood, axis=2) > neighbourhood.shape[2] // 2
        union = int(np.count_nonzero(output[:, :, slice_index] | consensus))
        if union == 0:
            continue
        score = float(np.count_nonzero(output[:, :, slice_index] ^ consensus) / union)
        scores[slice_index] = score
        outliers[slice_index] = score >= config.min_shape_disagreement_fraction

    # Join two suspicious parts separated by one unflagged slice. This is limited to
    # the detection mask; every resulting run must still pass the anatomical flank and
    # maximum-change gates below.
    outliers = ndi.binary_closing(outliers, structure=np.ones(3, dtype=bool))
    max_run_slices = max(1, int(np.floor(config.max_repair_run_mm / spacing[2])))
    repaired_runs: list[tuple[int, int]] = []
    skipped_runs: list[tuple[int, int, str]] = []
    added_voxels = 0
    removed_voxels = 0
    for start, stop in _true_runs(outliers):
        run_length = stop - start + 1
        if run_length > max_run_slices:
            skipped_runs.append((start, stop, "run_too_long"))
            continue
        left_index, right_index = start - 1, stop + 1
        if left_index < 0 or right_index >= output.shape[2]:
            skipped_runs.append((start, stop, "missing_flank"))
            continue
        if not (profile[left_index] and profile[right_index]):
            skipped_runs.append((start, stop, "outside_stable_brain_profile"))
            continue
        left = output[:, :, left_index]
        right = output[:, :, right_index]
        flank_dice = _dice(left, right)
        left_area = int(np.count_nonzero(left))
        right_area = int(np.count_nonzero(right))
        mean_area = max((left_area + right_area) / 2.0, np.finfo(float).eps)
        flank_area_change = abs(left_area - right_area) / mean_area
        if flank_dice < config.min_flank_dice:
            skipped_runs.append((start, stop, "flank_shape_mismatch"))
            continue
        if flank_area_change > config.max_flank_area_change_fraction:
            skipped_runs.append((start, stop, "flank_area_mismatch"))
            continue

        left_distance = _signed_distance(left, (float(spacing[0]), float(spacing[1])))
        right_distance = _signed_distance(right, (float(spacing[0]), float(spacing[1])))
        replacements: list[tuple[int, np.ndarray, int, int]] = []
        rejected_reason = ""
        for offset, slice_index in enumerate(range(start, stop + 1), start=1):
            weight = offset / (run_length + 1)
            interpolated = ((1.0 - weight) * left_distance + weight * right_distance) >= 0
            interpolated &= raw[:, :, slice_index]
            interpolated, _ = _largest_fully_connected_component(interpolated)
            if not interpolated.any():
                rejected_reason = "empty_interpolation"
                break
            current = output[:, :, slice_index]
            union = max(int(np.count_nonzero(current | interpolated)), 1)
            changed_fraction = np.count_nonzero(current ^ interpolated) / union
            if changed_fraction > config.max_slice_change_fraction:
                rejected_reason = "repair_exceeds_change_limit"
                break
            additions = int(np.count_nonzero(interpolated & ~current))
            removals = int(np.count_nonzero(current & ~interpolated))
            replacements.append((slice_index, interpolated, additions, removals))
        if rejected_reason:
            skipped_runs.append((start, stop, rejected_reason))
            continue
        for slice_index, replacement, additions, removals in replacements:
            output[:, :, slice_index] = replacement
            added_voxels += additions
            removed_voxels += removals
        repaired_runs.append((start, stop))

    output, final_disconnected = _largest_fully_connected_component(output)
    disconnected_removed += final_disconnected
    if np.any(output & ~raw):
        raise RuntimeError("M-seam cleanup created foreground outside the raw RS2 mask")
    changed_voxels = int(np.count_nonzero(candidate ^ output))
    scored_slices = tuple(
        (int(index), float(scores[index])) for index in np.flatnonzero(outliers)
    )
    report = MSeamCleanupReport(
        input_foreground_voxels=input_voxels,
        output_foreground_voxels=int(np.count_nonzero(output)),
        disconnected_voxels_removed=disconnected_removed,
        in_plane_island_voxels_removed=island_removed,
        in_plane_cleaned_slices=island_slices,
        candidate_outlier_slices=tuple(int(index) for index in np.flatnonzero(outliers)),
        outlier_score_by_slice=scored_slices,
        repaired_slice_runs=tuple(repaired_runs),
        skipped_slice_runs=tuple(skipped_runs),
        voxels_added_from_raw_rs2=added_voxels,
        voxels_removed_by_run_repair=removed_voxels,
        changed_voxels=changed_voxels,
        subset_of_raw_rs2=not bool(np.any(output & ~raw)),
        configuration=asdict(config),
    )
    return output, report


def refine_direct_seam(
    mask_rsa: np.ndarray,
    gaps: list[SliceGap],
    config: GapRefinementConfig,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Remove only RS2 voxels superior to each accepted dark-gap seam."""

    raw = np.asarray(mask_rsa, dtype=bool)
    output = raw.copy()
    accepted: list[int] = []
    rejected: list[int] = []
    for z, gap in enumerate(gaps):
        if not gap.valid:
            continue
        candidate = raw[:, :, z].copy()
        for x in range(gap.x_start, gap.x_stop):
            if np.isfinite(gap.seam[x]):
                candidate[x, int(np.floor(gap.seam[x])) + 1 :] = False
        candidate = _largest_component(candidate)
        if _accept_slice(raw[:, :, z], candidate, config):
            output[:, :, z] = candidate
            accepted.append(z)
        else:
            rejected.append(z)
    if len(accepted) < config.min_corrected_slices:
        output = raw.copy()
        accepted = []
    output = _largest_component(output)
    return output, _method_stats("rs2_m_seam", raw, output, gaps, accepted, rejected, config)


def refine_watershed(
    normalized_rsa: np.ndarray,
    mask_rsa: np.ndarray,
    gaps: list[SliceGap],
    spacing_rsa: tuple[float, float, float],
    config: GapRefinementConfig,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Use marker-controlled watershed to place the cut on an image gradient."""

    from skimage.segmentation import watershed

    image = np.asarray(normalized_rsa, dtype=np.float32)
    raw = np.asarray(mask_rsa, dtype=bool)
    output = raw.copy()
    accepted: list[int] = []
    rejected: list[int] = []
    for z, gap in enumerate(gaps):
        if not gap.valid:
            continue
        brain_seed, cap_seed, domain = _slice_markers(raw[:, :, z], gap, spacing_rsa[1], config)
        if not brain_seed.any() or not cap_seed.any():
            rejected.append(z)
            continue
        gradient = ndi.gaussian_gradient_magnitude(image[:, :, z], sigma=1.0)
        markers = np.zeros(raw.shape[:2], dtype=np.uint8)
        markers[brain_seed] = 1
        markers[cap_seed] = 2
        labels = watershed(gradient, markers=markers, mask=domain)
        candidate = raw[:, :, z].copy()
        candidate[domain & (labels == 2)] = False
        candidate = _largest_component(candidate)
        if _accept_slice(raw[:, :, z], candidate, config):
            output[:, :, z] = candidate
            accepted.append(z)
        else:
            rejected.append(z)
    if len(accepted) < config.min_corrected_slices:
        output = raw.copy()
        accepted = []
    output = _largest_component(output)
    return output, _method_stats("rs2_marker_watershed", raw, output, gaps, accepted, rejected, config)


def refine_random_walker(
    normalized_rsa: np.ndarray,
    mask_rsa: np.ndarray,
    gaps: list[SliceGap],
    spacing_rsa: tuple[float, float, float],
    config: GapRefinementConfig,
    beta: float = 180.0,
) -> tuple[np.ndarray, dict[str, Any]]:
    """Use marker-based random walker within small per-slice superior crops."""

    from skimage.segmentation import random_walker

    image = np.asarray(normalized_rsa, dtype=np.float32)
    raw = np.asarray(mask_rsa, dtype=bool)
    output = raw.copy()
    accepted: list[int] = []
    rejected: list[int] = []
    errors: dict[int, str] = {}
    for z, gap in enumerate(gaps):
        if not gap.valid:
            continue
        brain_seed, cap_seed, domain = _slice_markers(raw[:, :, z], gap, spacing_rsa[1], config)
        if not brain_seed.any() or not cap_seed.any():
            rejected.append(z)
            continue
        coordinates = np.argwhere(domain)
        x0, y0 = np.maximum(coordinates.min(axis=0) - 2, 0)
        x1, y1 = np.minimum(coordinates.max(axis=0) + 3, np.array(domain.shape))
        crop_domain = domain[x0:x1, y0:y1]
        labels = np.full(crop_domain.shape, -1, dtype=np.int8)
        labels[crop_domain] = 0
        labels[brain_seed[x0:x1, y0:y1]] = 1
        labels[cap_seed[x0:x1, y0:y1]] = 2
        try:
            result = random_walker(
                image[x0:x1, y0:y1, z], labels, beta=beta, mode="cg_j", tol=1e-3
            )
        except Exception as exc:  # Leave this experimental slice unchanged and report it.
            errors[z] = f"{type(exc).__name__}: {exc}"
            rejected.append(z)
            continue
        candidate = raw[:, :, z].copy()
        local_cap = crop_domain & (result == 2)
        candidate_crop = candidate[x0:x1, y0:y1]
        candidate_crop[local_cap] = False
        candidate = _largest_component(candidate)
        if _accept_slice(raw[:, :, z], candidate, config):
            output[:, :, z] = candidate
            accepted.append(z)
        else:
            rejected.append(z)
    if len(accepted) < config.min_corrected_slices:
        output = raw.copy()
        accepted = []
    output = _largest_component(output)
    stats = _method_stats("rs2_random_walker", raw, output, gaps, accepted, rejected, config)
    stats["beta"] = beta
    stats["slice_errors"] = errors
    return output, stats


def _method_stats(
    method: str,
    raw: np.ndarray,
    output: np.ndarray,
    gaps: list[SliceGap],
    accepted: list[int],
    rejected: list[int],
    config: GapRefinementConfig,
) -> dict[str, Any]:
    raw_count = int(np.count_nonzero(raw))
    output_count = int(np.count_nonzero(output))
    return {
        "method": method,
        "status": "corrected" if accepted else "unchanged_no_confident_correction",
        "raw_foreground_voxels": raw_count,
        "output_foreground_voxels": output_count,
        "removed_voxels": raw_count - output_count,
        "removed_fraction": (raw_count - output_count) / raw_count if raw_count else 0.0,
        "detected_gap_slices": [index for index, gap in enumerate(gaps) if gap.valid],
        "corrected_slices": accepted,
        "rejected_slices": rejected,
        "configuration": asdict(config),
    }


METHODS: dict[str, Callable[..., tuple[np.ndarray, dict[str, Any]]]] = {
    "rs2_m_seam": refine_direct_seam,
    "rs2_marker_watershed": refine_watershed,
    "rs2_random_walker": refine_random_walker,
}


In [ ]:
# Apply all three gated corrections to the same immutable raw prediction.
from dataclasses import asdict
import warnings

REFINEMENT_CONFIG = GapRefinementConfig(
    max_search_depth_mm=2.8,
    min_cap_thickness_mm=0.30,
    valley_window_mm=0.32,
    line_smoothing_mm=0.10,
    min_valley_contrast=0.10,
    min_confident_width_fraction=0.22,
    central_fraction=0.75,
    max_column_jump_mm=0.55,
    seed_margin_mm=0.24,
    min_slice_removed_fraction=0.01,
    max_slice_removed_fraction=0.35,
    min_corrected_slices=3,
)
RANDOM_WALKER_BETA = 180.0
REGULARITY_CONFIG = MaskRegularityConfig()
M_SEAM_CLEANUP_CONFIG = MSeamCleanupConfig()
METHOD_DESCRIPTIONS = {
    'rs2_m_seam': (
        'Direct removal superior to a detected dark M-shaped line plus conservative '
        'in-plane-island and short-run continuity cleanup'
    ),
    'rs2_marker_watershed': 'Marker-controlled watershed on the local T1 gradient',
    'rs2_random_walker': 'Marker-based random walker on local T1 intensity',
}
REFINEMENT_CACHE = {}
SUMMARY_ROWS = []

for case_index, case in enumerate(CASES, start=1):
    case_id, image_path = case['case_id'], case['image']
    print(f'[{case_index}/{len(CASES)}] Refining {case_id}')
    image_object = nib.load(str(image_path))
    raw_path = RESULTS / 'predictions' / 'rs2net_raw' / f'{case_id}_brain_mask.nii.gz'
    raw_native = np.asanyarray(nib.load(str(raw_path)).dataobj) > 0
    image_native = np.asanyarray(image_object.dataobj).astype(np.float32)
    image_rsa, orientation = native_to_rsa(image_native, image_object.affine)
    raw_rsa, mask_orientation = native_to_rsa(raw_native, image_object.affine)
    if orientation != mask_orientation:
        raise RuntimeError(f'Image/mask orientation mismatch for {case_id}')
    native_spacing = tuple(float(value) for value in image_object.header.get_zooms()[:3])
    spacing_rsa = tuple(native_spacing[axis] for axis in orientation['order'])
    normalized = robust_normalize(image_rsa, raw_rsa)
    gaps = detect_gap_volume(normalized, raw_rsa, spacing_rsa, REFINEMENT_CONFIG)
    raw_regularity = assess_mask_regularity(raw_rsa, spacing_rsa, REGULARITY_CONFIG)
    raw_metadata_path = RESULTS / 'metadata' / 'rs2net_raw' / f'{case_id}.json'
    raw_metadata = json.loads(raw_metadata_path.read_text())
    raw_metadata['regularity_qc'] = raw_regularity.to_dict()
    raw_metadata_path.write_text(json.dumps(raw_metadata, indent=2) + '\n')

    seam_before_cleanup, seam_stats = refine_direct_seam(
        raw_rsa, gaps, REFINEMENT_CONFIG
    )
    seam_mask, seam_cleanup = stabilize_m_seam_mask(
        seam_before_cleanup, raw_rsa, spacing_rsa, M_SEAM_CLEANUP_CONFIG
    )
    seam_stats['continuity_cleanup'] = seam_cleanup.to_dict()
    seam_stats['output_foreground_voxels'] = int(np.count_nonzero(seam_mask))
    seam_stats['removed_voxels'] = int(
        np.count_nonzero(raw_rsa) - np.count_nonzero(seam_mask)
    )
    seam_stats['removed_fraction'] = (
        seam_stats['removed_voxels'] / np.count_nonzero(raw_rsa)
    )
    watershed_mask, watershed_stats = refine_watershed(
        normalized, raw_rsa, gaps, spacing_rsa, REFINEMENT_CONFIG
    )
    results = {
        'rs2_m_seam': (seam_mask, seam_stats),
        'rs2_marker_watershed': (watershed_mask, watershed_stats),
    }
    if RUN_RANDOM_WALKER:
        with warnings.catch_warnings():
            warnings.filterwarnings('once', message='The probability range is outside')
            walker_mask, walker_stats = refine_random_walker(
                normalized, raw_rsa, gaps, spacing_rsa,
                REFINEMENT_CONFIG, beta=RANDOM_WALKER_BETA
            )
        results['rs2_random_walker'] = (walker_mask, walker_stats)

    surface_rsa = np.zeros_like(raw_rsa, dtype=np.uint8)
    for z, gap in enumerate(gaps):
        if not gap.valid:
            continue
        for x in range(gap.x_start, gap.x_stop):
            if np.isfinite(gap.seam[x]):
                surface_rsa[x, int(round(float(gap.seam[x]))), z] = 1
    surface_native = rsa_to_native(surface_rsa, orientation).astype(np.uint8)
    surface_path = RESULTS / 'diagnostics' / 'detected_gap' / f'{case_id}_detected_gap.nii.gz'
    if surface_native.any():
        save_mask_like(surface_native, image_object, surface_path)
    else:
        surface_path = None

    REFINEMENT_CACHE[case_id] = {
        'image_rsa': image_rsa, 'raw_rsa': raw_rsa, 'gaps': gaps,
        'spacing_rsa': spacing_rsa, 'orientation': orientation, 'results': results,
        'raw_regularity': raw_regularity,
    }
    for model_id, (candidate_rsa, stats) in results.items():
        regularity = assess_mask_regularity(
            candidate_rsa, spacing_rsa, REGULARITY_CONFIG
        )
        stats['regularity_qc'] = regularity.to_dict()
        candidate_native = rsa_to_native(candidate_rsa, orientation).astype(np.uint8)
        output_path = RESULTS / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        save_mask_like(candidate_native, image_object, output_path)
        removed_native = raw_native & ~candidate_native.astype(bool)
        removed_path = RESULTS / 'diagnostics' / 'removed' / model_id / f'{case_id}_removed_mask.nii.gz'
        if removed_native.any():
            save_mask_like(removed_native, image_object, removed_path)
        else:
            removed_path = None

        metadata_path = RESULTS / 'metadata' / model_id / f'{case_id}.json'
        metadata_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            'case_id': case_id, 'model_id': model_id, 'status': 'ok',
            'role': 'experimental automatic pre-label; human review required',
            'description': METHOD_DESCRIPTIONS[model_id],
            'input': relative(image_path), 'source_mask': relative(raw_path),
            'output': relative(output_path),
            'detected_gap': relative(surface_path) if surface_path else '',
            'removed_mask': relative(removed_path) if removed_path else '',
            'orientation_record': {
                'native_axis_codes': list(orientation['native_axis_codes']),
                'rsa_axis_order': orientation['order'], 'flips': orientation['flips'],
            },
            'input_sha256': sha256(image_path), 'source_mask_sha256': sha256(raw_path),
            'mask_sha256': sha256(output_path), 'refinement_statistics': stats,
            'scientific_warning': 'Candidate only; not approved and not valid for quantification without review.',
            **RS2_PROVENANCE, **RUNTIME,
        }
        metadata_path.write_text(json.dumps(payload, indent=2) + '\n')
        record(case_id, model_id, 'ok', image_path, output_path, metadata_path)
        SUMMARY_ROWS.append({
            'case_id': case_id, 'method': model_id, 'status': stats['status'],
            'detected_gap_slices': len(stats['detected_gap_slices']),
            'corrected_slices': len(stats['corrected_slices']),
            'removed_voxels': stats['removed_voxels'],
            'removed_percent': round(100.0 * stats['removed_fraction'], 3),
            'slice_errors': len(stats.get('slice_errors', {})),
            'regularity_warning_count': len(regularity.warnings),
            'regularity_warnings': ';'.join(regularity.warnings),
            'one_slice_outliers': len(regularity.one_slice_outlier_slices),
            'max_centroid_step_mm': round(regularity.max_centroid_step_mm, 4),
            'surface_area_mm2': round(regularity.surface_area_mm2, 4),
        })

with (RESULTS / 'refinement_summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(SUMMARY_ROWS[0]))
    writer.writeheader()
    writer.writerows(SUMMARY_ROWS)
print('Refinement candidates ready:', len(SUMMARY_ROWS))


In [ ]:
# Produce durable before/after montages and an interactive four-way viewer.
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import display
import ipywidgets as widgets

MODEL_ORDER = ['rs2net_raw', 'rs2_m_seam', 'rs2_marker_watershed']
if RUN_RANDOM_WALKER:
    MODEL_ORDER.append('rs2_random_walker')
MODEL_TITLES = {
    'rs2net_raw': 'Raw RS2-Net',
    'rs2_m_seam': 'M-seam cut',
    'rs2_marker_watershed': 'Marker watershed',
    'rs2_random_walker': 'Random walker',
}

def case_arrays(case_id):
    cache = REFINEMENT_CACHE[case_id]
    arrays = {'rs2net_raw': cache['raw_rsa']}
    arrays.update({key: value[0] for key, value in cache['results'].items()})
    return cache['image_rsa'], arrays

def display_limits(image, raw):
    values = image[raw & np.isfinite(image)]
    return tuple(np.percentile(values, (1, 99)))

def informative_slices(raw, arrays, count=5):
    removed = np.zeros_like(raw, dtype=bool)
    for key, candidate in arrays.items():
        if key != 'rs2net_raw':
            removed |= raw & ~candidate
    score = removed.sum(axis=(0, 1))
    choices = []
    for z in np.argsort(score)[::-1]:
        if score[z] == 0:
            break
        if all(abs(int(z) - other) >= 8 for other in choices):
            choices.append(int(z))
        if len(choices) == count:
            break
    occupied = np.flatnonzero(raw.any(axis=(0, 1)))
    fallback = np.linspace(occupied[0], occupied[-1], count + 2, dtype=int)[1:-1]
    for z in fallback:
        if all(abs(int(z) - other) >= 5 for other in choices):
            choices.append(int(z))
        if len(choices) == count:
            break
    return sorted(choices[:count])

qc_dir = RESULTS / 'qc'
qc_dir.mkdir(parents=True, exist_ok=True)
for case in CASES:
    case_id = case['case_id']
    image_rsa, arrays = case_arrays(case_id)
    raw = arrays['rs2net_raw']
    slices = informative_slices(raw, arrays)
    low, high = display_limits(image_rsa, raw)
    figure, axes = plt.subplots(
        len(MODEL_ORDER), len(slices), figsize=(3.2 * len(slices), 3.2 * len(MODEL_ORDER)),
        squeeze=False
    )
    for row, model_id in enumerate(MODEL_ORDER):
        candidate = arrays[model_id]
        for column, z in enumerate(slices):
            axis = axes[row, column]
            axis.imshow(image_rsa[:, :, z].T, origin='lower', cmap='gray', vmin=low, vmax=high)
            axis.contour(raw[:, :, z].T, levels=[0.5], colors='yellow', linewidths=0.7)
            if model_id != 'rs2net_raw':
                removed = raw[:, :, z] & ~candidate[:, :, z]
                axis.imshow(
                    np.ma.masked_where(~removed.T, removed.T), origin='lower',
                    cmap=ListedColormap(['magenta']), alpha=0.45, vmin=0, vmax=1
                )
                axis.contour(candidate[:, :, z].T, levels=[0.5], colors='cyan', linewidths=0.9)
            axis.set_title(f'{MODEL_TITLES[model_id]} | coronal {z}')
            axis.axis('off')
    figure.suptitle(
        f'{case_id}: yellow=raw RS2, cyan=corrected boundary, magenta=removed', fontsize=13
    )
    figure.tight_layout()
    output = qc_dir / f'{case_id}_rs2_refinement_comparison.png'
    figure.savefig(output, dpi=150, bbox_inches='tight')
    plt.close(figure)
print(f'Created {len(CASES)} QC montages in {qc_dir}')

def show_case(case_id, coronal_slice):
    image_rsa, arrays = case_arrays(case_id)
    z = min(int(coronal_slice), image_rsa.shape[2] - 1)
    raw = arrays['rs2net_raw']
    low, high = display_limits(image_rsa, raw)
    figure, axes = plt.subplots(2, 2, figsize=(11, 10))
    for axis, model_id in zip(axes.ravel(), MODEL_ORDER):
        candidate = arrays[model_id]
        axis.imshow(image_rsa[:, :, z].T, origin='lower', cmap='gray', vmin=low, vmax=high)
        axis.contour(raw[:, :, z].T, levels=[0.5], colors='yellow', linewidths=0.8)
        if model_id != 'rs2net_raw':
            removed = raw[:, :, z] & ~candidate[:, :, z]
            axis.imshow(
                np.ma.masked_where(~removed.T, removed.T), origin='lower',
                cmap=ListedColormap(['magenta']), alpha=0.45, vmin=0, vmax=1
            )
            axis.contour(candidate[:, :, z].T, levels=[0.5], colors='cyan', linewidths=1.0)
        axis.set_title(MODEL_TITLES[model_id])
        axis.axis('off')
    figure.suptitle(
        f'{case_id} | coronal slice {z} | yellow raw, cyan corrected, magenta removed'
    )
    figure.tight_layout()
    plt.show()

max_slices = max(REFINEMENT_CACHE[case['case_id']]['image_rsa'].shape[2] for case in CASES)
case_widget = widgets.Dropdown(
    options=[case['case_id'] for case in CASES], description='Case:'
)
slice_widget = widgets.IntSlider(
    value=min(125, max_slices - 1), min=0, max=max_slices - 1,
    step=1, description='Coronal:', continuous_update=False
)
display(widgets.HBox([case_widget, slice_widget]))
display(widgets.interactive_output(
    show_case, {'case_id': case_widget, 'coronal_slice': slice_widget}
))


In [ ]:
# Review the extent of every proposed correction before downloading.
import pandas as pd

summary = pd.DataFrame(SUMMARY_ROWS).sort_values(['case_id', 'method'])
display(summary)
print('\nInterpretation:')
print('- unchanged_no_confident_correction means the image-based safety gate retained raw RS2.')
print('- A large removed percentage is not automatically good; inspect the cortex for under-segmentation.')
print('- Regularity warnings prioritize review; they do not reject, repair, or approve a mask.')
print('- Compare the same method across all 10 cases. Do not select a method from one attractive slice.')
print('- These outputs are automatic candidates, not reviewed masks and not quantification inputs.')


In [ ]:
# Validate grids/subset behavior, record the experiment, and download one archive.
validation_rows = []
for case in CASES:
    case_id, image_path = case['case_id'], case['image']
    image_object = nib.load(str(image_path))
    image_native = np.asanyarray(image_object.dataobj)
    _, orientation = native_to_rsa(image_native, image_object.affine)
    native_spacing = tuple(float(value) for value in image_object.header.get_zooms()[:3])
    spacing_rsa = tuple(native_spacing[axis] for axis in orientation['order'])
    raw_path = RESULTS / 'predictions' / 'rs2net_raw' / f'{case_id}_brain_mask.nii.gz'
    raw_object = nib.load(str(raw_path))
    raw = np.asanyarray(raw_object.dataobj) > 0
    for model_id in MODEL_ORDER:
        mask_path = RESULTS / 'predictions' / model_id / f'{case_id}_brain_mask.nii.gz'
        mask_object = nib.load(str(mask_path))
        mask = np.asanyarray(mask_object.dataobj) > 0
        shape_ok = mask_object.shape == image_object.shape
        affine_ok = np.allclose(mask_object.affine, image_object.affine, rtol=1e-5, atol=1e-5)
        binary_ok = set(np.unique(np.asanyarray(mask_object.dataobj))).issubset({0, 1})
        nonempty_ok = bool(mask.any() and not mask.all())
        subset_of_raw = bool(not np.any(mask & ~raw))
        mask_rsa, _ = native_to_rsa(mask, image_object.affine)
        regularity = assess_mask_regularity(
            mask_rsa, spacing_rsa, REGULARITY_CONFIG
        )
        if not all((shape_ok, affine_ok, binary_ok, nonempty_ok, subset_of_raw)):
            raise ValueError(f'Validation failed for {case_id} {model_id}')
        validation_rows.append({
            'case_id': case_id, 'model_id': model_id, 'shape_ok': shape_ok,
            'affine_ok': affine_ok, 'binary_ok': binary_ok,
            'nonempty_ok': nonempty_ok, 'subset_of_raw_rs2': subset_of_raw,
            'foreground_voxels': int(mask.sum()),
            'regularity_warning_count': len(regularity.warnings),
            'regularity_warnings': ';'.join(regularity.warnings),
            'max_adjacent_area_change_fraction': round(
                regularity.max_adjacent_area_change_fraction, 6
            ),
            'max_centroid_step_mm': round(regularity.max_centroid_step_mm, 6),
            'surface_area_mm2': round(regularity.surface_area_mm2, 6),
            'compactness': round(regularity.compactness, 6),
        })

with (RESULTS / 'validation_summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(validation_rows[0]))
    writer.writeheader()
    writer.writerows(validation_rows)

readme = (
    'RS2-Net T1-guided refinement experiment\n\n'
    'Every mask in this archive is an automatic pre-label. Human review is mandatory.\n'
    '`rs2net_raw` is immutable. The other folders contain experimental corrections.\n\n'
    'QC colors:\n'
    '  yellow  raw RS2 boundary\n'
    '  cyan    corrected boundary\n'
    '  magenta voxels removed from raw RS2\n\n'
    'After downloading, review all masks with:\n'
    '  conda run -n lys-bbb python scripts/brain_extraction/review_colab_results.py \\\n'
    '    ~/Downloads/t1_brain_extraction_rs2_refinement_results.zip\n\n'
    'Inspect the full anterior-posterior extent and specifically check for lost superior cortex.\n'
    'Regularity warnings are review aids only; they do not approve or repair a mask.\n'
    'Do not use an automatic candidate for quantification until it has been accepted or corrected.\n'
)
(RESULTS / 'README.txt').write_text(readme)
run_metadata = {
    'created_at': utc_now(),
    'purpose': 'Experimental T1-guided correction of superior skull in RS2-Net masks',
    'case_count': len(CASES), 'models': MODEL_ORDER,
    'refinement_configuration': asdict(REFINEMENT_CONFIG),
    'regularity_configuration': asdict(REGULARITY_CONFIG),
    'random_walker_enabled': RUN_RANDOM_WALKER,
    'random_walker_beta': RANDOM_WALKER_BETA if RUN_RANDOM_WALKER else None,
    'rs2_provenance': RS2_PROVENANCE, 'runtime': RUNTIME,
    'approval_status': 'none; all outputs require human review',
}
(RESULTS / 'run_metadata.json').write_text(json.dumps(run_metadata, indent=2) + '\n')

archive_base = Path('/content/t1_brain_extraction_rs2_refinement_results')
archive_path = Path(shutil.make_archive(
    str(archive_base), 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name
))
print(f'Validated {len(validation_rows)} image/mask combinations.')
print('Archive:', archive_path, f'({archive_path.stat().st_size / 1e6:.1f} MB)')
files.download(str(archive_path))
